# Installing timm which has all 4 architectures and segmentation models for UNet++

In [ ]:
# ============================================================
# DEEPFAKE DETECTION - COMPLETE TRAINING PIPELINE
# Models: EfficientNetB0, Xception, UNet++, Swin Transformer
# Datasets: Celeb-DF + FaceForensics++

!pip install timm segmentation-models-pytorch onnx onnxruntime -q

 **SETTING UP FOLDER PATHS**

In [ ]:
# ============================================================
# SECTION 1 - ALL IMPORTS
# ============================================================

import os
import gc
import csv
import time
import math
import random
import shutil
import subprocess
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from torch.amp import GradScaler

import timm
import segmentation_models_pytorch as smp

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report
)

# setting seeds for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# device setup
if torch.cuda.is_available():
    device = 'cuda'
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = 'cpu'
    print("No GPU found. Using CPU.")

In [ ]:
# ============================================================
# SECTION 2 - ALL PATHS
# ============================================================

# input dataset paths - read only
celeb_base      = '/kaggle/input/datasets/princefiebor/celeb-df-dataset/Celeb Dataset/Celeb-DF'
celeb_real_path = os.path.join(celeb_base, 'Celeb-real')
celeb_yt_path   = os.path.join(celeb_base, 'YouTube-real')
celeb_fake_path = os.path.join(celeb_base, 'Celeb-synthesis')
celeb_test_file = os.path.join(celeb_base, 'List_of_testing_videos.txt')

face_base      = '/kaggle/input/datasets/princefiebor/face-dataset/face++dataset'
face_real_path = os.path.join(face_base, 'ffpp_real')
face_fake_path = os.path.join(face_base, 'ffpp_fake')

# output paths
frames_dir       = '/kaggle/working/Frames'
models_save_path = '/kaggle/working/Saved_Models'
manifest_dir     = '/kaggle/working/Manifests'
results_dir      = '/kaggle/working/Results'

for folder in [frames_dir, models_save_path, manifest_dir, results_dir]:
    os.makedirs(folder, exist_ok=True)

# confirming all input paths exist before starting
all_good = True
for name, path in {
    'Celeb-real'    : celeb_real_path,
    'YouTube-real'  : celeb_yt_path,
    'Celeb-synthesis': celeb_fake_path,
    'Test list'     : celeb_test_file,
    'FF++ real'     : face_real_path,
    'FF++ fake'     : face_fake_path
}.items():
    exists = os.path.exists(path)
    print(f"{name}: {'OK' if exists else 'MISSING'}")
    if not exists:
        all_good = False

print(f"\nAll paths ready: {all_good}")

In [ ]:
# ============================================================
# SECTION 3 - VIDEO ID BASED SPLITTING
# This is the most critical section.
# We NEVER split at frame level because frames from the same
# video are highly correlated. If frame 1 is in train and
# frame 2 is in val, the model cheats and gets fake 99% accuracy.
# Instead we split at VIDEO level - all frames from one video
# stay together in the same split.
# ============================================================

# reading official celeb-df test list
with open(celeb_test_file, 'r') as f:
    raw_lines = f.read().splitlines()

# cleaning to get just filenames
celeb_official_test = set()
for line in raw_lines:
    celeb_official_test.add(os.path.basename(line.strip()))

print(f"Official Celeb-DF test videos: {len(celeb_official_test)}")

# building a master video registry
# each entry: {video_id, source_path, label, split}
# label: 1=real, 0=fake
# split: train, val, test
video_registry = []

# ---- CELEB-DF REAL VIDEOS ----
for vid in os.listdir(celeb_real_path):
    if vid.endswith(('.mp4', '.avi', '.mov')):
        split = 'test' if vid in celeb_official_test else 'trainval'
        video_registry.append({
            'video_id'   : f"celebdf_real_{vid}",
            'source_path': os.path.join(celeb_real_path, vid),
            'label'      : 1,
            'split'      : split,
            'dataset'    : 'celebdf'
        })

# ---- CELEB-DF YOUTUBE REAL VIDEOS ----
for vid in os.listdir(celeb_yt_path):
    if vid.endswith(('.mp4', '.avi', '.mov')):
        split = 'test' if vid in celeb_official_test else 'trainval'
        video_registry.append({
            'video_id'   : f"celebdf_yt_{vid}",
            'source_path': os.path.join(celeb_yt_path, vid),
            'label'      : 1,
            'split'      : split,
            'dataset'    : 'celebdf'
        })

# ---- CELEB-DF FAKE VIDEOS ----
for vid in os.listdir(celeb_fake_path):
    if vid.endswith(('.mp4', '.avi', '.mov')):
        split = 'test' if vid in celeb_official_test else 'trainval'
        video_registry.append({
            'video_id'   : f"celebdf_fake_{vid}",
            'source_path': os.path.join(celeb_fake_path, vid),
            'label'      : 0,
            'split'      : split,
            'dataset'    : 'celebdf'
        })

# ---- FACEFORENSICS++ REAL VIDEOS ----
# no official test list so we do 70/10/20 split by video id
all_ffpp_real = sorted([
    f for f in os.listdir(face_real_path)
    if f.endswith(('.mp4', '.avi', '.mov'))
])

random.seed(42)
random.shuffle(all_ffpp_real)

ffpp_real_train_cut = int(0.70 * len(all_ffpp_real))
ffpp_real_val_cut   = int(0.80 * len(all_ffpp_real))

for idx, vid in enumerate(all_ffpp_real):
    if idx < ffpp_real_train_cut:
        split = 'trainval'
    elif idx < ffpp_real_val_cut:
        split = 'val_only'
    else:
        split = 'test'
    video_registry.append({
        'video_id'   : f"ffpp_real_{vid}",
        'source_path': os.path.join(face_real_path, vid),
        'label'      : 1,
        'split'      : split,
        'dataset'    : 'ffpp'
    })

# ---- FACEFORENSICS++ FAKE VIDEOS ----
all_ffpp_fake = sorted([
    f for f in os.listdir(face_fake_path)
    if f.endswith(('.mp4', '.avi', '.mov'))
])

random.seed(42)
random.shuffle(all_ffpp_fake)

ffpp_fake_train_cut = int(0.70 * len(all_ffpp_fake))
ffpp_fake_val_cut   = int(0.80 * len(all_ffpp_fake))

for idx, vid in enumerate(all_ffpp_fake):
    if idx < ffpp_fake_train_cut:
        split = 'trainval'
    elif idx < ffpp_fake_val_cut:
        split = 'val_only'
    else:
        split = 'test'
    video_registry.append({
        'video_id'   : f"ffpp_fake_{vid}",
        'source_path': os.path.join(face_fake_path, vid),
        'label'      : 0,
        'split'      : split,
        'dataset'    : 'ffpp'
    })

# splitting trainval videos 80/20 by video id for train vs val
# all frames from one video stay in the same split
trainval_videos = [v for v in video_registry if v['split'] == 'trainval']
random.seed(42)
random.shuffle(trainval_videos)

train_cut = int(0.80 * len(trainval_videos))

for idx, v in enumerate(trainval_videos):
    v['split'] = 'train' if idx < train_cut else 'val'

# adding val_only videos as val
for v in video_registry:
    if v['split'] == 'val_only':
        v['split'] = 'val'

# counting splits
train_vids = [v for v in video_registry if v['split'] == 'train']
val_vids   = [v for v in video_registry if v['split'] == 'val']
test_vids  = [v for v in video_registry if v['split'] == 'test']

train_real = sum(1 for v in train_vids if v['label'] == 1)
train_fake = sum(1 for v in train_vids if v['label'] == 0)
val_real   = sum(1 for v in val_vids   if v['label'] == 1)
val_fake   = sum(1 for v in val_vids   if v['label'] == 0)
test_real  = sum(1 for v in test_vids  if v['label'] == 1)
test_fake  = sum(1 for v in test_vids  if v['label'] == 0)

print(f"Train videos : {len(train_vids)} (Real: {train_real}, Fake: {train_fake})")
print(f"Val videos   : {len(val_vids)}   (Real: {val_real},   Fake: {val_fake})")
print(f"Test videos  : {len(test_vids)}  (Real: {test_real},  Fake: {test_fake})")
print(f"Total videos : {len(video_registry)}")

In [ ]:
# ============================================================
# SECTION 4 - FRAME EXTRACTION WITH VIDEO ID TRACKING
# Extracts every 10th frame from each video.
# Each frame filename contains the video_id so we always know
# which video it came from - critical for video level evaluation.
# The csv manifest records frame_path, label, video_id, split.
# ============================================================

FRAME_SKIP = 10

def extract_frames_for_split(video_list, split_name):

    split_frame_dir = os.path.join(frames_dir, split_name)
    os.makedirs(split_frame_dir, exist_ok=True)

    frame_records = []

    for video_info in tqdm(video_list, desc=f"Extracting {split_name} frames"):

        video_id   = video_info['video_id']
        src_path   = video_info['source_path']
        label      = video_info['label']
        safe_id    = video_id.replace('/', '_').replace(' ', '_')

        # checking if this video was already extracted
        first_frame = os.path.join(split_frame_dir, f"{safe_id}_f0.jpg")
        if os.path.exists(first_frame):
            # collecting existing frames for this video
            existing = [
                f for f in os.listdir(split_frame_dir)
                if f.startswith(safe_id) and f.endswith('.jpg')
            ]
            for fname in existing:
                frame_records.append({
                    'frame_path': os.path.join(split_frame_dir, fname),
                    'label'     : label,
                    'video_id'  : video_id,
                    'split'     : split_name
                })
            continue

        cap = cv2.VideoCapture(src_path)
        if not cap.isOpened():
            print(f"Could not open: {src_path}")
            continue

        frame_idx  = 0
        saved      = 0
        success, frame = cap.read()

        while success:
            if frame_idx % FRAME_SKIP == 0:
                # frame name includes video_id and frame index
                frame_name = f"{safe_id}_f{frame_idx}.jpg"
                save_path  = os.path.join(split_frame_dir, frame_name)
                cv2.imwrite(save_path, frame)
                frame_records.append({
                    'frame_path': save_path,
                    'label'     : label,
                    'video_id'  : video_id,
                    'split'     : split_name
                })
                saved += 1
            success, frame = cap.read()
            frame_idx += 1

        cap.release()

    print(f"{split_name}: {len(frame_records)} frames from {len(video_list)} videos")
    return frame_records


# extracting frames for each split separately
# this guarantees no video appears in two splits
print("Extracting training frames...")
train_records = extract_frames_for_split(train_vids, 'train')

print("\nExtracting validation frames...")
val_records   = extract_frames_for_split(val_vids,   'val')

print("\nExtracting test frames...")
test_records  = extract_frames_for_split(test_vids,  'test')

# saving manifests to csv
def save_manifest(records, csv_path):
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['frame_path', 'label', 'video_id', 'split'])
        writer.writeheader()
        writer.writerows(records)
    print(f"Manifest saved: {csv_path} ({len(records)} rows)")

train_csv = os.path.join(manifest_dir, 'train_manifest.csv')
val_csv   = os.path.join(manifest_dir, 'val_manifest.csv')
test_csv  = os.path.join(manifest_dir, 'test_manifest.csv')

save_manifest(train_records, train_csv)
save_manifest(val_records,   val_csv)
save_manifest(test_records, test_csv)

# checking disk space after extraction
result = subprocess.run('df -h /kaggle/working',
                        capture_output=True, text=True, shell=True)
print(result.stdout)

In [ ]:
# ============================================================
# SECTION 5 - DATASET CLASS WITH VIDEO ID AWARENESS
# Reads from the manifest csv.
# Lazy loading - only the requested image is read at a time.
# Supports WeightedRandomSampler for balanced batches.
# ============================================================

class DeepfakeDataset(Dataset):

    def __init__(self, csv_path, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.transform = transform

        # counting labels for sampler weights
        self.labels = self.df['label'].values

        real_count = (self.labels == 1).sum()
        fake_count = (self.labels == 0).sum()
        print(f"Dataset loaded from {csv_path}")
        print(f"  Real frames: {real_count}, Fake frames: {fake_count}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['frame_path']
        label    = row['label']
        video_id = row['video_id']

        # lazy loading - only this one image is opened now
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = torch.tensor([label], dtype=torch.float32)
        return img, label, video_id


# collate function that handles the video_id string in the batch
def collate_fn(batch):
    imgs      = torch.stack([item[0] for item in batch])
    labels    = torch.stack([item[1] for item in batch])
    video_ids = [item[2] for item in batch]
    return imgs, labels, video_ids

In [ ]:
# ============================================================
# SECTION 6 - TRANSFORMS
# Training uses heavy augmentation to force the model to focus
# on facial manipulation artifacts rather than background or
# identity features.
# Testing uses only resize and normalize - no augmentation.
# ============================================================

# imagenet normalization values - matches pretrained model expectations
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# heavy augmentation for training
# forces model to learn manipulation artifacts not background features
train_transforms = transforms.Compose([

    # resizing to 224x224 to match all model input sizes
    transforms.Resize((224, 224)),

    # randomly flipping left to right
    transforms.RandomHorizontalFlip(p=0.5),

    # randomly rotating up to 15 degrees
    transforms.RandomRotation(degrees=15),

    # randomly adjusting brightness contrast saturation hue
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),

    # randomly applying gaussian blur to simulate compression artifacts
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),

    # converting to tensor
    transforms.ToTensor(),

    # normalizing with imagenet statistics
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# simple transforms for validation and testing - no augmentation
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [ ]:
# ============================================================
# SECTION 7 - DATALOADERS WITH BALANCED BATCH SAMPLING
# WeightedRandomSampler ensures each batch has roughly 50/50
# real and fake frames regardless of dataset imbalance.
# This prevents the model from learning to guess the majority class.
# ============================================================

def make_loader(csv_path, transform, batch_size, is_train=False):

    dataset = DeepfakeDataset(csv_path, transform=transform)

    if is_train:
        # computing sample weights so each batch is balanced 50/50
        # real frames get weight inversely proportional to their count
        # fake frames get weight inversely proportional to their count
        labels       = dataset.labels
        class_counts = np.bincount(labels)
        class_weights = 1.0 / class_counts

        # each sample gets the weight of its class
        sample_weights = torch.tensor(
            [class_weights[l] for l in labels], dtype=torch.float
        )

        # sampler draws samples with replacement using these weights
        sampler = WeightedRandomSampler(
            weights     = sample_weights,
            num_samples = len(sample_weights),
            replacement = True
        )

        loader = DataLoader(
            dataset,
            batch_size  = batch_size,
            sampler     = sampler,
            num_workers = 0,
            collate_fn  = collate_fn,
            pin_memory  = True if device == 'cuda' else False
        )
    else:
        loader = DataLoader(
            dataset,
            batch_size  = batch_size,
            shuffle     = False,
            num_workers = 0,
            collate_fn  = collate_fn,
            pin_memory  = True if device == 'cuda' else False
        )

    return loader, dataset


BATCH_SIZE = 32

train_loader, train_dataset = make_loader(train_csv, train_transforms,     BATCH_SIZE, is_train=True)
val_loader,   val_dataset   = make_loader(val_csv,   val_test_transforms,  BATCH_SIZE, is_train=False)
test_loader,  test_dataset  = make_loader(test_csv,  val_test_transforms,  BATCH_SIZE, is_train=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# ============================================================
# SECTION 8 - UNET++ CLASSIFIER
# UNet++ is a segmentation architecture adapted here for
# binary classification by replacing the decoder output with
# a global average pooling and a fully connected classification head.
# 320 is the correct output channel count for efficientnet-b0 encoder.
# ============================================================

class UNetPPClassifier(nn.Module):

    def __init__(self):
        super(UNetPPClassifier, self).__init__()

        # unet++ with efficientnet-b0 encoder
        # imagenet weights give it a head start on visual features
        self.unetpp = smp.UnetPlusPlus(
            encoder_name    = 'efficientnet-b0',
            encoder_weights = 'imagenet',
            in_channels     = 3,
            classes         = 1
        )

        # global average pooling reduces spatial feature maps to single values
        self.gap = nn.AdaptiveAvgPool2d(1)

        # dropout before classifier to reduce overfitting
        self.dropout = nn.Dropout(p=0.3)

        # 320 matches the output channels of efficientnet-b0 last encoder layer
        self.fc = nn.Linear(320, 1)

    def forward(self, x):
        # extracting deep features from the encoder
        features     = self.unetpp.encoder(x)
        last_feature = features[-1]

        # pooling and flattening
        pooled  = self.gap(last_feature)
        flat    = pooled.view(pooled.size(0), -1)
        dropped = self.dropout(flat)
        output  = self.fc(dropped)
        return output

In [ ]:
# ============================================================
# SECTION 9 - WARMUP LEARNING RATE SCHEDULER
# For the first few epochs we gradually increase the learning rate
# from a very small value up to the target learning rate.
# This prevents large gradient steps early in training from
# permanently damaging the model's ability to learn subtle artifacts.
# After warmup, we switch to cosine annealing which smoothly
# reduces the learning rate over the remaining epochs.
# ============================================================

class WarmupCosineScheduler:

    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 base_lr=1e-6, target_lr=1e-4):

        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr       = base_lr
        self.target_lr     = target_lr
        self.current_epoch = 0

    def step(self):

        self.current_epoch += 1

        if self.current_epoch <= self.warmup_epochs:
            # linearly increasing lr from base_lr to target_lr during warmup
            progress = self.current_epoch / self.warmup_epochs
            lr = self.base_lr + progress * (self.target_lr - self.base_lr)
        else:
            # cosine annealing after warmup
            progress = (self.current_epoch - self.warmup_epochs) / (
                self.total_epochs - self.warmup_epochs
            )
            lr = self.target_lr * 0.5 * (1 + math.cos(math.pi * progress))

        # applying new lr to all parameter groups
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

        return lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

In [ ]:
# ============================================================
# SECTION 10 - TRAINING FUNCTION
# Uses AMP mixed precision to cut GPU memory roughly in half.
# Uses gradient accumulation for large models like Swin Transformer.
# Saves a checkpoint every epoch for safety.
# Implements early stopping to avoid wasted compute.
# ============================================================

def train_model(model, model_name, train_loader, val_loader,
                num_epochs=15, target_lr=1e-4, warmup_epochs=3,
                accumulation_steps=1, patience=4):

    model     = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-6, weight_decay=1e-4
    )

    # warmup + cosine scheduler
    scheduler = WarmupCosineScheduler(
        optimizer, warmup_epochs, num_epochs,
        base_lr=1e-6, target_lr=target_lr
    )

    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler(device)

    train_losses, train_accs = [], []
    val_losses,   val_accs   = [], []

    best_val_loss  = float('inf')
    best_val_auc   = 0.0
    stop_counter   = 0
    best_epoch     = 0

    for epoch in range(num_epochs):

        current_lr = scheduler.get_lr()
        print(f"\n{model_name} | Epoch {epoch+1}/{num_epochs} | LR: {current_lr:.2e}")

        # ---- TRAINING PHASE ----
        model.train()
        running_loss = 0.0
        correct      = 0
        total        = 0
        optimizer.zero_grad()

        for batch_idx, (imgs, labels, _) in enumerate(train_loader):

            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()

            # amp mixed precision forward pass
            with torch.amp.autocast(device):
                outputs = model(imgs)
                loss    = criterion(outputs, labels) / accumulation_steps

            # scaled backward pass
            scaler.scale(loss).backward()

            # stepping optimizer every accumulation_steps batches
            if (batch_idx + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item() * accumulation_steps
            predicted     = (torch.sigmoid(outputs) > 0.5).float()
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)

        # handling remaining gradients from last partial accumulation
        if len(train_loader) % accumulation_steps != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # stepping the lr scheduler after each epoch
        scheduler.step()

        epoch_train_loss = running_loss / len(train_loader)
        epoch_train_acc  = correct / total
        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc)

        # ---- VALIDATION PHASE ----
        model.eval()
        val_loss    = 0.0
        val_correct = 0
        val_total   = 0
        val_preds   = []
        val_labels  = []
        val_probs   = []

        with torch.no_grad():
            for val_imgs, val_lbls, _ in val_loader:
                val_imgs = val_imgs.to(device, non_blocking=True)
                val_lbls = val_lbls.to(device, non_blocking=True).float()

                with torch.amp.autocast(device):
                    val_outputs = model(val_imgs)
                    v_loss      = criterion(val_outputs, val_lbls)

                val_loss += v_loss.item()
                probs     = torch.sigmoid(val_outputs)
                preds     = (probs > 0.5).float()

                val_correct += (preds == val_lbls).sum().item()
                val_total   += val_lbls.size(0)
                val_preds.extend(preds.cpu().numpy().flatten())
                val_labels.extend(val_lbls.cpu().numpy().flatten())
                val_probs.extend(probs.cpu().numpy().flatten())

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc  = val_correct / val_total
        epoch_val_auc  = roc_auc_score(val_labels, val_probs)

        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc)

        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Val Loss  : {epoch_val_loss:.4f} | Val Acc  : {epoch_val_acc:.4f} | Val AUC: {epoch_val_auc:.4f}")

        # saving checkpoint every epoch for safety
        # if session crashes we can always resume from last epoch
        checkpoint_path = os.path.join(
            models_save_path, f'{model_name}_epoch{epoch+1}.pt'
        )
        torch.save({
            'epoch'               : epoch + 1,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss'            : epoch_val_loss,
            'val_auc'             : epoch_val_auc,
            'train_losses'        : train_losses,
            'val_losses'          : val_losses
        }, checkpoint_path)

        # keeping only the best checkpoint to save disk space
        # deleting the previous best if a new best is found
        if epoch_val_auc > best_val_auc:
            # deleting old best checkpoint
            old_best = os.path.join(models_save_path, f'{model_name}_BEST.pt')
            if os.path.exists(old_best):
                os.remove(old_best)

            # saving new best
            torch.save(model.state_dict(),
                       os.path.join(models_save_path, f'{model_name}_BEST.pt'))

            best_val_auc  = epoch_val_auc
            best_val_loss = epoch_val_loss
            best_epoch    = epoch + 1
            stop_counter  = 0
            print(f"New best model saved at epoch {epoch+1} (AUC: {best_val_auc:.4f})")
        else:
            stop_counter += 1
            print(f"No improvement. Early stop counter: {stop_counter}/{patience}")

            # deleting non-best epoch checkpoints to save disk space
            # keeping only the most recent and the best
            if epoch > 0:
                prev_checkpoint = os.path.join(
                    models_save_path, f'{model_name}_epoch{epoch}.pt'
                )
                if os.path.exists(prev_checkpoint):
                    os.remove(prev_checkpoint)

            if stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                print(f"Best epoch was {best_epoch} with AUC: {best_val_auc:.4f}")
                break

    return model, train_losses, train_accs, val_losses, val_accs, best_val_auc

In [ ]:
# ============================================================
# SECTION 11 - VIDEO LEVEL EVALUATION
# We trained on frames but deepfake detection is judged
# per video not per frame.
# For each video we average all its frame probabilities
# to get a single video level prediction.
# This is the correct way to evaluate deepfake detectors.
# ============================================================

def evaluate_video_level(model, model_name, test_loader, results_csv):

    model.eval()

    # collecting frame level predictions grouped by video id
    video_probs  = defaultdict(list)
    video_labels = {}

    # gpu inference timing
    gpu_times = []

    with torch.no_grad():
        for imgs, labels, video_ids in test_loader:

            imgs   = imgs.to(device, non_blocking=True)
            start  = time.time()

            with torch.amp.autocast(device):
                outputs = model(imgs)

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end = time.time()
            gpu_times.append(end - start)

            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            lbls  = labels.numpy().flatten()

            # grouping predictions by video id
            for vid_id, prob, lbl in zip(video_ids, probs, lbls):
                video_probs[vid_id].append(prob)
                video_labels[vid_id] = lbl

    avg_gpu_ms = (sum(gpu_times) / len(gpu_times)) * 1000

    # cpu inference timing on 5 batches
    model_cpu = model.to('cpu')
    cpu_times = []
    with torch.no_grad():
        for batch_idx, (imgs, _, __) in enumerate(test_loader):
            if batch_idx >= 5:
                break
            start = time.time()
            model_cpu(imgs)
            end = time.time()
            cpu_times.append(end - start)
    avg_cpu_ms = (sum(cpu_times) / len(cpu_times)) * 1000
    model = model.to(device)

    # aggregating frame probabilities to video level
    # averaging all frame predictions for each video
    all_video_ids    = list(video_probs.keys())
    all_video_probs  = np.array([np.mean(video_probs[v]) for v in all_video_ids])
    all_video_labels = np.array([video_labels[v] for v in all_video_ids])
    all_video_preds  = (all_video_probs > 0.5).astype(int)

    # computing video level metrics
    acc       = accuracy_score(all_video_labels, all_video_preds)
    precision = precision_score(all_video_labels, all_video_preds, zero_division=0)
    recall    = recall_score(all_video_labels, all_video_preds, zero_division=0)
    f1        = f1_score(all_video_labels, all_video_preds, zero_division=0)
    auc_score = roc_auc_score(all_video_labels, all_video_probs)

    print(f"\n{'='*50}")
    print(f"{model_name} - VIDEO LEVEL EVALUATION")
    print(f"{'='*50}")
    print(f"Videos evaluated : {len(all_video_ids)}")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision        : {precision:.4f}")
    print(f"Recall           : {recall:.4f}")
    print(f"F1 Score         : {f1:.4f}")
    print(f"AUC              : {auc_score:.4f}")
    print(f"GPU ms/batch     : {avg_gpu_ms:.2f}")
    print(f"CPU ms/batch     : {avg_cpu_ms:.2f}")

    print("\nClassification Report (Video Level):")
    print(classification_report(
        all_video_labels, all_video_preds,
        target_names=["Fake", "Real"]
    ))

    # confusion matrix
    cm = confusion_matrix(all_video_labels, all_video_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Fake", "Real"],
                yticklabels=["Fake", "Real"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name} - Confusion Matrix (Video Level)")
    plt.show()

    # roc curve
    fpr, tpr, _ = roc_curve(all_video_labels, all_video_probs)
    roc_auc_val = auc(fpr, tpr)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color='steelblue', label=f'AUC = {roc_auc_val:.4f}')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{model_name} - ROC Curve (Video Level)")
    plt.legend()
    plt.show()

    # logging to results csv
    file_exists = os.path.exists(results_csv)
    with open(results_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                'model', 'accuracy', 'precision', 'recall',
                'f1', 'auc', 'gpu_ms_per_batch', 'cpu_ms_per_batch',
                'videos_evaluated'
            ])
        writer.writerow([
            model_name, acc, precision, recall, f1, auc_score,
            avg_gpu_ms, avg_cpu_ms, len(all_video_ids)
        ])

    print(f"Results logged to {results_csv}")

    return {
        'model': model_name, 'accuracy': acc, 'precision': precision,
        'recall': recall, 'f1': f1, 'auc': auc_score,
        'video_probs': all_video_probs, 'video_labels': all_video_labels
    }

In [ ]:
# ============================================================
# SECTION 12 - MODEL REGISTRY AND SEQUENTIAL TRAINING
# All 4 models trained one at a time.
# Each model is fully cleared from GPU memory before next loads.
# Swin Transformer replaces vanilla ViT because Swin has
# hierarchical attention with local windows which makes it
# much more suitable for deepfake detection than vanilla ViT.
# ============================================================

results_csv  = os.path.join(results_dir, 'benchmark_results.csv')
all_results  = {}

# wiping old results file
if os.path.exists(results_csv):
    os.remove(results_csv)

model_registry = {

    'EfficientNetB0': {
        # lightweight efficient model - fastest inference
        # good baseline for real time chrome extension use
        'builder'           : lambda: timm.create_model(
            'efficientnet_b0', pretrained=True, num_classes=1
        ),
        'epochs'            : 15,
        'target_lr'         : 1e-4,
        'warmup_epochs'     : 3,
        'accumulation_steps': 1,
        'patience'          : 4
    },

    'Xception': {
        # strong depthwise separable convolution architecture
        # shown to be effective for deepfake detection in literature
        'builder'           : lambda: timm.create_model(
            'xception', pretrained=True, num_classes=1
        ),
        'epochs'            : 15,
        'target_lr'         : 1e-4,
        'warmup_epochs'     : 3,
        'accumulation_steps': 1,
        'patience'          : 4
    },

    'UNetPP': {
        # segmentation architecture adapted for classification
        # can localize manipulation boundaries in addition to classifying
        'builder'           : None,  # custom class defined above
        'epochs'            : 15,
        'target_lr'         : 1e-4,
        'warmup_epochs'     : 3,
        'accumulation_steps': 2,
        'patience'          : 4
    },

    'SwinTransformer': {
        # swin transformer replaces vanilla vit
        # hierarchical shifted window attention is much better suited
        # for spatial artifact detection than vanilla vit patches
        # also trains more stably than vanilla vit from pretrained weights
        'builder'           : lambda: timm.create_model(
            'swin_base_patch4_window7_224', pretrained=True, num_classes=1
        ),
        'epochs'            : 15,
        'target_lr'         : 5e-5,
        'warmup_epochs'     : 5,
        # accumulation_steps=4 simulates batch 128 while using only 32 in memory
        'accumulation_steps': 4,
        'patience'          : 4
    }
}

for model_name, config in model_registry.items():

    print(f"\n{'='*60}")
    print(f"STARTING: {model_name}")
    print(f"{'='*60}")

    # checking disk space before loading each model
    space = subprocess.run('df -h /kaggle/working',
                           capture_output=True, text=True, shell=True)
    print(space.stdout)

    # building the model
    if model_name == 'UNetPP':
        model = UNetPPClassifier()
    else:
        model = config['builder']()

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters    : {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # training
    model, t_loss, t_acc, v_loss, v_acc, best_auc = train_model(
        model,
        model_name,
        train_loader,
        val_loader,
        num_epochs          = config['epochs'],
        target_lr           = config['target_lr'],
        warmup_epochs       = config['warmup_epochs'],
        accumulation_steps  = config['accumulation_steps'],
        patience            = config['patience']
    )

    # plotting training curves
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(8, 8))
    ax1.plot(t_loss, label='Train', color='steelblue')
    ax1.plot(v_loss, label='Val',   color='tomato')
    ax1.set_title(f'{model_name} - Loss vs Epochs')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax2.plot(t_acc, label='Train', color='steelblue')
    ax2.plot(v_acc, label='Val',   color='tomato')
    ax2.set_title(f'{model_name} - Accuracy vs Epochs')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    plt.tight_layout()
    plt.show()


   # loading best checkpoint for evaluation
    best_weights_path = os.path.join(models_save_path, f'{model_name}_BEST.pt')

    # if no improvement happened during training the BEST file might not exist
    # in that case we save the current model state as the best
    if not os.path.exists(best_weights_path):
        torch.save(model.state_dict(), best_weights_path)
        print(f"No best checkpoint found - saving current state as best")

    # rebuilding the model architecture for evaluation
    if model_name == 'UNetPP':
        eval_model = UNetPPClassifier()
    else:
        eval_model = config['builder']()

    # loading the best saved weights into the fresh model
    eval_model.load_state_dict(
        torch.load(best_weights_path, map_location=device, weights_only=True)
    )
    eval_model = eval_model.to(device)

    # video level evaluation on test set
    result = evaluate_video_level(eval_model, model_name, test_loader, results_csv)
    all_results[model_name] = result

    # cleaning up epoch checkpoints to free disk space
    for f in os.listdir(models_save_path):
        if f.startswith(model_name) and 'epoch' in f:
            os.remove(os.path.join(models_save_path, f))
            print(f"Deleted: {f}")

    # checking space after cleanup
    space = subprocess.run('df -h /kaggle/working',
                           capture_output=True, text=True, shell=True)
    print(space.stdout)

    # freeing all memory before next model loads
    del model
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{model_name} cleared from GPU memory\n")

print("\nAll 4 models trained and evaluated!")

In [ ]:
# ============================================================
# SECTION 13 - BENCHMARK COMPARISON TABLE
# Produces the comprehensive comparison table required for
# your documentation and supervisor submission.
# Covers all required metrics: accuracy, precision, recall,
# F1, AUC, and inference time on GPU and CPU.
# ============================================================

# checking results file exists before reading
if not os.path.exists(results_csv):
    print("No results file found - no models completed evaluation yet")
else:
    results_df = pd.read_csv(results_csv)
    print("\nFINAL BENCHMARK TABLE")
    print("=" * 80)
    print(results_df.to_string(index=False))
    print("=" * 80)

print("\nFINAL BENCHMARK TABLE")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

# finding best model by AUC score
best_row  = results_df.loc[results_df['auc'].idxmax()]
best_name = best_row['model']
print(f"\nBest Model: {best_name}")
print(f"  AUC      : {best_row['auc']:.4f}")
print(f"  F1 Score : {best_row['f1']:.4f}")
print(f"  Accuracy : {best_row['accuracy']:.4f}")

# grouped bar chart comparing all 4 models on all metrics
metrics     = ['accuracy', 'precision', 'recall', 'f1', 'auc']
model_names = results_df['model'].tolist()
x           = np.arange(len(model_names))
width       = 0.15

fig, ax = plt.subplots(figsize=(14, 6))
for idx, metric in enumerate(metrics):
    bars = ax.bar(x + idx * width, results_df[metric], width, label=metric)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(model_names)
ax.set_ylabel('Score')
ax.set_title('Deepfake Detection Benchmark - All Models All Metrics')
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

# inference time comparison
fig, ax = plt.subplots(figsize=(10, 5))
x    = np.arange(len(model_names))
w    = 0.35
ax.bar(x - w/2, results_df['gpu_ms_per_batch'], w, label='GPU ms/batch', color='steelblue')
ax.bar(x + w/2, results_df['cpu_ms_per_batch'], w, label='CPU ms/batch', color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('Milliseconds per Batch')
ax.set_title('Inference Time Comparison - GPU vs CPU')
ax.legend()
plt.tight_layout()
plt.show()

# saving the benchmark table as csv for your documentation
results_df.to_csv(os.path.join(results_dir, 'final_benchmark.csv'), index=False)
print(f"Benchmark table saved to {results_dir}/final_benchmark.csv")

In [ ]:
# ============================================================
# SECTION 14 - EXPORTING BEST MODEL TO ONNX
# The best model by AUC is exported to ONNX format.
# This is the model that will be loaded by app.py for
# real time deepfake detection in the Chrome extension.
# input.1 is the input node name expected by app.py
# ============================================================

print(f"Exporting best model: {best_name}")

# rebuilding the best model architecture
if best_name == 'EfficientNetB0':
    best_model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=1)
elif best_name == 'Xception':
    best_model = timm.create_model('xception', pretrained=False, num_classes=1)
elif best_name == 'UNetPP':
    best_model = UNetPPClassifier()
else:
    best_model = timm.create_model(
        'swin_base_patch4_window7_224', pretrained=False, num_classes=1
    )

# loading the best saved weights
best_weights_path = os.path.join(models_save_path, f'{best_name}_BEST.pt')
best_model.load_state_dict(
    torch.load(best_weights_path, map_location=device, weights_only=True)
)
best_model = best_model.to(device)
best_model.eval()

# creating a dummy input to trace the model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path   = os.path.join(models_save_path, 'FinalModel.onnx')

torch.onnx.export(
    best_model,
    dummy_input,
    onnx_path,
    export_params      = True,
    opset_version      =14,
    do_constant_folding= True,
    input_names        = ['input.1'],
    output_names       = ['output']
)

print(f"Best model ({best_name}) exported to ONNX: {onnx_path}")

# verifying the exported model loads correctly
import onnxruntime as ort
ort_session = ort.InferenceSession(onnx_path)
test_input  = np.random.randn(1, 3, 224, 224).astype(np.float32)
ort_output  = ort_session.run(None, {'input.1': test_input})
print(f"ONNX model verified. Output shape: {ort_output[0].shape}")
print("Pipeline complete!")